# Error of SLAE-CS after removing residues with a large predicted error

Keeps the residues whose SLAE-ERR predicted error is at most X ppm and computes their MAE. The ensemble standard deviation, random selection and an oracle keep the same number of residues. 95% ranges from a protein-cluster bootstrap.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PRED_PATH = Path("../../data/analysis/Fig5/predictions_N_ensemble_10seed_sorted_centered_err.json")
ENSEMBLE_PATH = Path("../../data/analysis/Fig4/results/predictions_N_seed_ensemble_10seed.json")
FIG = Path("Figs")

# ---- CONFIG ----
THRESHOLDS = np.round(np.arange(0.5, 2.55, 0.05), 2)   # X (ppm)
TABLE_X = (0.6, 0.8, 1.0, 1.2, 1.5, 2.0)               # rows of the printed table
TOP_TICKS_X = (0.6, 1.0, 1.5, 2.0, 2.5)                # X values labeled with the kept fraction
MIN_RESIDUES = 100                                     # thresholds keeping fewer residues are dropped
N_BOOT = 2000
N_RANDOM = 2000                                        # random selections
SEED = 0
TAG = ""                                               # suffix of the output files
COLORS = {"SLAE-ERR": "#3F729B", "Ensemble std": "#D1793F", "Random": "#999999", "Oracle": "#555555"}

## Load residues

In [ ]:
raw = json.load(open(PRED_PATH))
ens = json.load(open(ENSEMBLE_PATH))
rows = []
for entry in raw["pred"]:
    valid = np.array(raw["mask"][entry], bool) & np.array(raw["oof_mask"][entry], bool)
    assert len(ens["std"][entry]) == len(valid), entry
    rows.append(pd.DataFrame(dict(entry=entry, pred=np.array(raw["pred"][entry], float)[valid],
                                  std=np.array(ens["std"][entry], float)[valid],
                                  obs=np.array(raw["oof_resid"][entry], float)[valid])))
df = pd.concat(rows, ignore_index=True)
assert np.isfinite(df[["pred", "std", "obs"]].to_numpy()).all() and (df.obs >= 0).all()
u, std, obs = df.pred.to_numpy(), df["std"].to_numpy(), df.obs.to_numpy()
codes, entries = pd.factorize(df.entry)
print(f"{len(df)} residues, {len(entries)} proteins; all residues: MAE {obs.mean():.3f} ppm; "
      f"predicted error {u.min():.3f}-{u.max():.3f} ppm")

## Threshold on the predicted error

In [ ]:
kept_n = np.array([(u <= x).sum() for x in THRESHOLDS])
X = THRESHOLDS[kept_n >= MIN_RESIDUES]
print(f"thresholds {X[0]:.2f}-{X[-1]:.2f} ppm ({len(X)}); dropped below {MIN_RESIDUES} residues: {THRESHOLDS[kept_n < MIN_RESIDUES].tolist()}")

# per protein and threshold: residues kept and sum of |error|, for SLAE-ERR (N, A) and the ensemble std (N_std, A_std)
n_entries = len(entries)
N, A, N_std, A_std = (np.zeros((n_entries, len(X))) for _ in range(4))
std_order = np.argsort(std, kind="stable")   # lowest std first
std_cutoff = np.zeros(len(X))
for j, x in enumerate(X):
    k = u <= x
    N[:, j] = np.bincount(codes[k], minlength=n_entries)
    A[:, j] = np.bincount(codes[k], obs[k], minlength=n_entries)
    picked = std_order[:k.sum()]             # the same number of residues as SLAE-ERR
    std_cutoff[j] = std[picked[-1]]
    N_std[:, j] = np.bincount(codes[picked], minlength=n_entries)
    A_std[:, j] = np.bincount(codes[picked], obs[picked], minlength=n_entries)
total = np.bincount(codes, minlength=n_entries).astype(float)


def curves(weight):
    """Kept fraction, SLAE-ERR MAE, ensemble std MAE and their difference at each X, for proteins
    weighted by the number of times they were drawn."""
    n = weight @ N
    err_mae, std_mae = (weight @ A) / n, (weight @ A_std) / (weight @ N_std)
    return n / (weight @ total), err_mae, std_mae, err_mae - std_mae


cov, mae, std_mae, delta = curves(np.ones(n_entries))
rng = np.random.default_rng(SEED)
boot = [curves(np.bincount(rng.integers(0, n_entries, n_entries), minlength=n_entries).astype(float))
        for _ in range(N_BOOT)]
boot = np.array(boot)                        # resample, metric, X
lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
delta_p = np.maximum(2 * np.minimum((boot[:, 3] <= 0).mean(0), (boot[:, 3] >= 0).mean(0)), 1 / N_BOOT)

T = pd.DataFrame(dict(X=X, n_kept=(N.sum(0)).astype(int), kept_fraction=cov, kept_fraction_lo=lo[0], kept_fraction_hi=hi[0],
                      mae=mae, mae_lo=lo[1], mae_hi=hi[1], std_cutoff=std_cutoff,
                      std_mae=std_mae, std_mae_lo=lo[2], std_mae_hi=hi[2],
                      difference=delta, difference_lo=lo[3], difference_hi=hi[3], difference_p=delta_p))

# read back: the per-protein sums must give the same values as a direct calculation
for j, x in enumerate(X):
    k = u <= x
    assert abs(cov[j] - k.mean()) < 1e-12 and abs(mae[j] - obs[k].mean()) < 1e-9
    assert abs(std_mae[j] - obs[std_order[:k.sum()]].mean()) < 1e-9 and N_std[:, j].sum() == k.sum()
assert np.all(np.diff(cov) >= 0)

## Random selection and oracle

At each X, these keep the same number of residues as SLAE-ERR, as does the ensemble std above (lowest std first). Random: proteins are resampled, then residues are drawn at random, N_RANDOM times; the band is the 2.5th to 97.5th percentile of the MAE. Oracle: the residues with the smallest observed error; it cannot be used in practice and marks the limit of any selection.

In [ ]:
ranked = np.sort(obs)
mean_kept = np.cumsum(ranked) / np.arange(1, len(ranked) + 1)
idx = np.maximum(np.round(cov * len(obs)).astype(int), 1) - 1
T["oracle_mae"] = mean_kept[idx]

rng_random = np.random.default_rng(SEED)
parts = [obs[codes == i] for i in range(n_entries)]
random_mae = np.empty((N_RANDOM, len(X)))
for b in range(N_RANDOM):
    picked = np.concatenate([parts[i] for i in rng_random.integers(0, n_entries, n_entries)])
    running = np.cumsum(rng_random.permutation(picked))
    n = np.maximum(np.round(cov * len(picked)).astype(int), 1)
    random_mae[b] = running[n - 1] / n
T["random_mae"] = random_mae.mean(0)
T["random_mae_lo"], T["random_mae_hi"] = np.percentile(random_mae, [2.5, 97.5], axis=0)
assert np.abs(T.random_mae - obs.mean()).max() < 0.01    # the expected MAE of a random subset is that of all residues

FIG.mkdir(exist_ok=True)
path = FIG / f"T_error_threshold{TAG}.csv"
T.to_csv(path, index=False)
assert len(pd.read_csv(path)) == len(T)
print("saved", path.name)
show = ["X", "n_kept", "kept_fraction", "mae", "mae_lo", "mae_hi", "std_mae", "std_mae_lo", "std_mae_hi",
        "difference", "difference_lo", "difference_hi", "random_mae", "random_mae_lo", "random_mae_hi", "oracle_mae"]
T[T.X.isin(TABLE_X)][show].round(3)

## Figure

In [ ]:
from matplotlib.collections import LineCollection
from matplotlib.legend_handler import HandlerTuple
from matplotlib.lines import Line2D


def dash_segments(ax, x, y, dash=5.5, gap=2.4):
    """Cut a curve into separate straight dashes of equal length (points on the final axes), in data coordinates."""
    to_points = 72 / ax.figure.dpi
    xy = ax.transData.transform(np.column_stack([x, y])) * to_points
    fine = np.linspace(0, 1, 2000)
    along = np.r_[0, np.cumsum(np.hypot(*np.diff(xy, axis=0).T))]
    grid = fine * along[-1]
    pts = np.column_stack([np.interp(grid, along, xy[:, 0]), np.interp(grid, along, xy[:, 1])])
    segments, start = [], 0.0
    while start < along[-1]:
        end = min(start + dash, along[-1])
        inside = (grid >= start) & (grid <= end)
        p = pts[inside] / to_points
        if len(p) > 1:
            segments.append(ax.transData.inverted().transform(p))
        start = end + gap
    return segments


def set_style(base=14, family="Arial"):
    plt.rcParams.update({"font.family": family, "font.size": base,
        "axes.titlesize": base, "axes.labelsize": base, "xtick.labelsize": base - 2,
        "ytick.labelsize": base - 2, "legend.fontsize": base - 2, "svg.fonttype": "none"})

set_style(base=15, family="Arial")

fig, a = plt.subplots(figsize=(5, 5))

a.fill_between(X, T.random_mae_lo, T.random_mae_hi, color=COLORS["Random"], alpha=0.18, lw=0)
a.plot(X, T.random_mae, color=COLORS["Random"], lw=2, label="Random")
a.fill_between(X, lo[2], hi[2], color=COLORS["Ensemble std"], alpha=0.18, lw=0)
a.plot(X, std_mae, color=COLORS["Ensemble std"], lw=2, label="Ensemble std")
a.fill_between(X, lo[1], hi[1], color=COLORS["SLAE-ERR"], alpha=0.18, lw=0)
a.plot(X, mae, color=COLORS["SLAE-ERR"], lw=2, label="SLAE-ERR")
a.set_xlabel("SLAE-ERR prediction cutoff (ppm)")
a.set_ylabel("MAE of kept residues (ppm)")
a.set_xlim(X[0], X[-1])
a.set_ylim(0, None)
y_limits = a.get_ylim()
# legend entry of the oracle: three short solid lines, so its dashes are real geometry too
dash_key = tuple(Line2D([], [], color=COLORS["Oracle"], lw=1.5) for _ in range(3))
handles, labels = a.get_legend_handles_labels()
a.legend(handles + [dash_key], labels + ["Oracle"], frameon=False, loc="lower right",
         handler_map={tuple: HandlerTuple(ndivide=None, pad=0.35)})
top = a.twiny()
top.set_xlim(a.get_xlim())
ticks = [x for x in TOP_TICKS_X if X[0] <= x <= X[-1]]
top.set_xticks(ticks)
# one decimal when rounding would wrongly show 100%
kept = [(u <= x).mean() for x in ticks]
top.set_xticklabels([f"{k:.1%}" if 0.995 <= k < 1 else f"{k:.0%}" for k in kept])
top.set_xlabel("Residues kept")

a.spines["right"].set_visible(False)
top.spines["right"].set_visible(False)
fig.tight_layout()

# oracle: each dash is its own line in the SVG, not a stroke-dasharray, so dash length scales with the figure.
# Added after tight_layout because the dash lengths are measured on the final axes.
a.add_collection(LineCollection(dash_segments(a, X, T.oracle_mae.to_numpy()), colors=COLORS["Oracle"], linewidths=1.5,
                                capstyle="butt"))
a.set_xlim(X[0], X[-1])
a.set_ylim(*y_limits)
fig.savefig(FIG / f"T_error_threshold{TAG}.svg", bbox_inches="tight")
fig.savefig(FIG / f"T_error_threshold{TAG}.png", dpi=300, bbox_inches="tight")
print("saved figure")